# 📊 골든크로스 기반 포트폴리오 최적화 스크리너

---

## 🧭 개요

이 노트북은 **골든크로스(Golden Cross)** 신호가 발생한 코스피 종목을 실시간으로 탐지하고,
해당 종목들을 대상으로 **포트폴리오 최적화**까지 수행하는 퀀트 스크리닝 도구입니다.

> 💡 **골든크로스란?** 단기 이동평균선(예: 20일)이 장기 이동평균선(예: 60일)을 아래에서 위로 돌파하는 시점으로, 일반적으로 상승 추세 전환의 기술적 신호로 해석됩니다.

---

## 🔄 분석 흐름 (Pipeline)



---

## 📦 사용 라이브러리

| 라이브러리 | 용도 |
|---|---|
|  | 주가 데이터 수집 |
|  | 코스피 종목 리스트 조회 |
|  | 한국 주식 시장 데이터 보조 |
|  /  | 수치 계산 및 데이터 처리 |
|  | 시각화 |

---

## ⚙️ 주요 설정값

| 항목 | 값 | 설명 |
|---|---|---|
| 무위험 수익률 () | 3% | 샤프지수 계산 기준 |
| 분석 기간 | 최근 3년 | 수익률 및 공분산 추정 |
| 시뮬레이션 횟수 | 15,000회 | 몬테카를로 포트폴리오 탐색 |

---

## 🚀 실행 방법

1. **상단부터 순서대로 셀을 실행**합니다. ( 또는 전체 실행)
2. 골든크로스 종목이 자동으로 탐지되며, 데이터 수집 → 분석 → 시각화까지 자동 진행됩니다.
3. 종목 탐지에 실패하면 오류 메시지를 확인하고, 네이버 금융 접속 환경 또는 사이트 구조 변경 여부를 점검하세요.

---

## ⚠️ 유의사항

- 이 노트북은 **교육 및 연구 목적**으로 제작되었으며, 실제 투자 의사결정에 직접 활용하는 것은 권장하지 않습니다.
- 골든크로스는 후행 지표로, 모든 상승 전환을 보장하지 않습니다.
- 분석 결과는 과거 수익률 기반이며, 미래 수익률을 보장하지 않습니다.


In [ ]:
import yfinance as yf
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime, timedelta

# 1. 설정
rf_rate = 0.03 # 무위험 수익률 3%
end_date = datetime.now().strftime('%Y-%m-%d')
start_date = (datetime.now() - timedelta(days=365*3)).strftime('%Y-%m-%d')

print(f"[분석 실행 일자: {end_date}]")
print(f"분석 기간: {start_date} ~ {end_date}")

## 골든크로스 스크리닝 (Golden Cross Screening)

In [ ]:
!pip install -U finance-datareader

In [ ]:
import requests

print("1. 네이버 금융에서 골든크로스 종목 추출 중...")
url = 'https://finance.naver.com/sise/item_gold.naver'
header = {'User-Agent': 'Mozilla/5.0'}

try:
    # pandas를 이용해 테이블 데이터 추출
    res = requests.get(url, headers=header)
    df_list = pd.read_html(res.text)
    # 골든크로스 테이블 선택 (보통 두 번째 테이블)
    df_gold = df_list[1].dropna(subset=['종목명'])

    # 상위 5개 종목 코드 추출
    top_5_codes = df_gold.head(5)['종목명'].values.tolist()
    # FinanceDataReader를 이용해 종목명을 코드로 변환하거나 직접 추출 (여기서는 테이블 내 링크나 코드가 필요하므로 StockListing 활용)
    import FinanceDataReader as fdr
    all_stocks = fdr.StockListing('KRX')

    tickers = []
    for name in top_5_codes:
        row = all_stocks[all_stocks['Name'] == name]
        if not row.empty:
            code = row.iloc[0]['Code']
            suffix = '.KS' if row.iloc[0]['Market'] == 'KOSPI' else '.KQ'
            tickers.append(code + suffix)

    print(f"✅ 포착된 상위 5개 종목: {tickers}")

    # 데이터 수집
    data = yf.download(tickers, start=start_date, end=end_date)['Close']
    ret_daily = np.log(data / data.shift(1)).dropna()
    ret_annual = ret_daily.mean() * 252
    cov_annual = ret_daily.cov() * 252
    print("\n🚀 데이터 분석 준비 완료.")

except Exception as e:
    print(f"❌ 오류 발생: {e}")
    print("사이트 구조 변경이나 연결 문제일 수 있습니다.")

### 🔍 골든크로스 포착 종목 확인

위의 전 종목 스캔 과정에서 포착된 종목 리스트를 확인합니다. 만약 종목이 너무 적게 나온다면 스캔이 완료될 때까지 기다리거나, 분석 대상을 조정할 수 있습니다.

In [ ]:
if 'golden_cross_tickers' in locals() and golden_cross_tickers:
    print(f"✅ 현재까지 포착된 골든크로스 종목 ({len(golden_cross_tickers)}개):")
    print(golden_cross_tickers)

    # 분석 대상 tickers를 포착된 종목으로 업데이트
    if len(golden_cross_tickers) >= 1:
        tickers = golden_cross_tickers
        print("\n🚀 분석 대상(tickers)이 포착된 종목으로 업데이트되었습니다.")
else:
    print("❌ 현재까지 포착된 골든크로스 종목이 없습니다. 스캔이 진행 중이거나 오늘 조건에 맞는 종목이 없을 수 있습니다.")

In [ ]:
!pip install pykrx

### 전 종목 리스트 자동 추출

언제든지 실행 버튼을 누르면 현재 시장에 상장된 종목 리스트를 실시간으로 불러옵니다. 이 리스트를 활용해 골든크로스 종목을 스캔합니다.

In [ ]:
import FinanceDataReader as fdr

# 코스피 상장 종목 리스트 가져오기
df_kospi = fdr.StockListing('KOSPI')

# yfinance 형식(종목코드.KS)으로 변환
full_kospi_tickers = [code + '.KS' for code in df_kospi['Code']]

print(f"현재 코스피 상장 종목 수: {len(full_kospi_tickers)}개")
print(f"상위 5개 종목 예시: {full_kospi_tickers[:5]}")

## 개별 종목 분석 및 상관관계

In [ ]:
if 'ret_annual' in locals() and not ret_annual.empty:
    individual_volatility = np.sqrt(np.diag(cov_annual))
    # 무위험 수익률 3% 반영
    individual_sharpe_ratio = (ret_annual - 0.03) / individual_volatility

    analysis_df = pd.DataFrame({
        'Expected Return': ret_annual,
        'Volatility': individual_volatility,
        'Sharpe Ratio': individual_sharpe_ratio
    })

    print("\n" + "="*55)
    print(" [네이버 골든크로스 상위 종목 투자 지표] ".center(55, "="))
    display(analysis_df.style.format({'Expected Return': '{:.2%}', 'Volatility': '{:.2%}', 'Sharpe Ratio': '{:.4f}'}))

    print("\n" + "="*55)
    print(" [종목 간 상관계수 행렬] ".center(55, "="))
    display(ret_daily.corr().style.format('{:.4f}'))
else:
    print("분석할 종목 데이터가 없습니다. 상단 스캔 셀을 먼저 실행해주세요.")

In [ ]:
# 포트폴리오 최적화 (몬테카를로 시뮬레이션)
n_ports = 15000
port_ret = []
port_vol = []
port_weights = []
sharpe_ratio = []

np.random.seed(42)
for _ in range(n_ports):
    weights = np.random.random(len(tickers))
    weights /= np.sum(weights)

    ret = np.dot(weights, ret_annual)
    vol = np.sqrt(np.dot(weights.T, np.dot(cov_annual, weights)))

    port_ret.append(ret)
    port_vol.append(vol)
    port_weights.append(weights)
    sharpe_ratio.append((ret - 0.03) / vol)

results = pd.DataFrame({'Returns': port_ret, 'Volatility': port_vol, 'Sharpe': sharpe_ratio})

# 시각화 및 최적 비중 출력
plt.figure(figsize=(10, 7))
plt.scatter(results['Volatility'], results['Returns'], c=results['Sharpe'], cmap='viridis', alpha=0.3)
plt.colorbar(label='Sharpe Ratio')

max_sharpe_idx = results['Sharpe'].idxmax()
max_sharpe = results.iloc[max_sharpe_idx]
best_weights = port_weights[max_sharpe_idx]

plt.scatter(max_sharpe['Volatility'], max_sharpe['Returns'], color='red', marker='*', s=200, label='Max Sharpe Ratio')
plt.title(f'Portfolio Optimization: Golden Cross Top Stocks')
plt.xlabel('Volatility')
plt.ylabel('Returns')
plt.legend()
plt.show()

print("-" * 50)
print(f"🚀 최적 포트폴리오 투자 비중 (Max Sharpe: {max_sharpe['Sharpe']:.2f})")
for i, ticker in enumerate(tickers):
    print(f"   - {ticker}: {best_weights[i]:.2%}")
print("-" * 50)

### 📈 포착 종목 최근 주가 흐름 분석
선정된 종목들의 최근 1년간 상대적 주가 추이를 비교합니다.

In [ ]:
import matplotlib.pyplot as plt

# 최근 1년 데이터로 한정하여 시각화
plot_data = data.last('365D')

# 시작가를 100으로 정규화 (Relative Price)
normalized_df = (plot_data / plot_data.iloc[0] * 100)

plt.figure(figsize=(14, 7))
for column in normalized_df.columns:
    plt.plot(normalized_df.index, normalized_df[column], label=column, linewidth=2)

plt.title('Golden Cross Tickers: Relative Price Trend (Last 1 Year)', fontsize=16)
plt.xlabel('Date')
plt.ylabel('Normalized Price (Start = 100)')
plt.legend(loc='upper left')
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()

### 두 시뮬레이션 결과 비교

두 버전의 최대 샤프 지수 차이를 확인하기 위해 각 시뮬레이션에서 도출된 최적값을 비교합니다.

In [ ]:
comparison = pd.DataFrame({
    'Max Sharpe Portfolio': [max_sharpe['Sharpe'], max_sharpe['Returns'], max_sharpe['Volatility']],
    'Min Volatility Portfolio': [min_vol['Sharpe'], min_vol['Returns'], min_vol['Volatility']]
}, index=['Sharpe Ratio', 'Return', 'Volatility'])

display(comparison.style.format({
    'Sharpe Ratio': '{:.4f}',
    'Return': '{:.2%}',
    'Volatility': '{:.2%}'
}))